# Train BarkNet on Google Colab

Train **YOLOv8s-seg** and **ConvexMask R50-FPN** on BarkNet data0.

**Runtime**: Enable GPU (Runtime → Change runtime type → T4 GPU)

## 1. Clone / upload your Bark project

In [ ]:
# Option A: Clone from Git
!git clone https://github.com/YOUR_USERNAME/Bark.git /content/Bark

# Option B: Upload a zip of your Bark folder, then:
# !unzip -q Bark.zip -d /content/

%cd /content/Bark

## 2. YOLOv8s-seg training (~15–20 min on T4)

In [ ]:
!pip install -q ultralytics

from pathlib import Path
from ultralytics import YOLO

BARK = Path("/content/Bark")
DATA = BARK / "BarkNetYOLO" / "data0" / "dataset.yaml"

model = YOLO("yolov8s-seg.pt")
model.train(
    data=str(DATA),
    epochs=100,
    batch=8,
    imgsz=640,
    project="/content/results",
    name="barknet_data0_yolov8s_seg",
)

print("Done! Best model:", "/content/results/barknet_data0_yolov8s_seg/weights/best.pt")

## 3. ConvexMask R50-FPN training (~45–90 min on T4)

In [ ]:
!pip install -q torch torchvision opencv-python pycocotools

# Clone ConvexMask
!git clone -q https://github.com/rcondat/convexmask.git /content/Bark/convexmask_repo

# Download ResNet50 backbone (if not present)
import torch
from torchvision.models import resnet50, ResNet50_Weights
Path("/content/Bark/convexmask_repo/weights").mkdir(parents=True, exist_ok=True)
backbone_path = "/content/Bark/convexmask_repo/weights/resnet50-19c8e357.pth"
if not Path(backbone_path).exists():
    m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    torch.save(m.state_dict(), backbone_path)
    print("Saved ResNet50 backbone")

# Run training script (patches config and trains)
!cd /content/Bark && python scripts/train_barknet_convexmask.py --convexmask_repo /content/Bark/convexmask_repo

## 4. Download trained models

In [ ]:
from google.colab import files

# YOLOv8
files.download("/content/results/barknet_data0_yolov8s_seg/weights/best.pt")

# ConvexMask (path may vary)
# files.download("/content/Bark/convexmask_repo/weights/convex_barknet_r50/best_checkpoint.pth")